In [ ]:
print(5)

In [ ]:
! pip install transformers torch faiss-cpu pandas openpyxl

In [ ]:
"""
icd10_faiss_excel.py
ICD-10 Semantic Search using SapBERT + FAISS
--------------------------------------------
Requirements:
    pip install transformers torch faiss-cpu pandas openpyxl

Input Excel Format:
    Code | Description
    A000 | Cholera due to Vibrio cholerae 01, biovar cholerae
"""

import pandas as pd
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModel

# ---------------------
# CONFIG
# ---------------------
MODEL_NAME = "sentence-transformers/embeddinggemma-300m-medical"
FAISS_INDEX_PATH = "icd10_faiss.index"
EXCEL_METADATA = "icd10_metadata.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32


# ---------------------
# MODEL SETUP
# ---------------------
def load_model_and_tokenizer(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(DEVICE)
    return tokenizer, model


def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)


def embed_texts(texts, tokenizer, model, batch_size=BATCH_SIZE):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    # L2 normalization for cosine similarity
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


# ---------------------
# INDEX CREATION
# ---------------------
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product for cosine similarity (after normalization)
    index.add(embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved at {FAISS_INDEX_PATH}")
    return index


# ---------------------
# ICD-10 INDEX PIPELINE
# ---------------------
def index_icd10_excel(excel_path):
    df = pd.read_excel(excel_path, dtype=str).fillna("")
    if not {"Code", "Description"}.issubset(df.columns):
        raise ValueError("Excel must contain columns: 'Code' and 'Description'")

    texts = df["Description"].tolist()  # ✅ Only use description for embeddings

    tokenizer, model = load_model_and_tokenizer()
    print("🔹 Generating embeddings...")
    embs = embed_texts(texts, tokenizer, model)

    print("🔹 Building FAISS index...")
    index = build_faiss_index(embs)

    # Save metadata for lookup
    df["faiss_id"] = range(len(df))
    df.to_excel(EXCEL_METADATA, index=False)
    print(f"✅ Metadata saved to {EXCEL_METADATA}")

    # Create mapping: id -> {code, description}
    metadata_map = {
        i: {"code": row["Code"], "description": row["Description"]}
        for i, row in df.iterrows()
    }

    return index, tokenizer, model, metadata_map


# ---------------------
# SEARCH FUNCTION
# ---------------------
def search_icd10(query, index, tokenizer, model, metadata_map, top_k=10, threshold=0.35):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        if score >= threshold:
            meta = metadata_map[idx]
            results.append(
                {
                    "ICD_Code": meta["code"],
                    "Description": meta["description"],
                    "Score (cosine)": round(float(score), 4),
                }
            )
    return results


# ---------------------
# MAIN USAGE EXAMPLE
# ---------------------


In [ ]:
"""
icd10_faiss_excel.py
ICD-10 Semantic Search using SapBERT + FAISS
--------------------------------------------
Requirements:
    pip install transformers torch faiss-cpu pandas openpyxl

Input Excel Format:
    Code | Description
    A000 | Cholera due to Vibrio cholerae 01, biovar cholerae
"""

import pandas as pd
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModel

# ---------------------
# CONFIG
# ---------------------
MODEL_NAME = "sentence-transformers/embeddinggemma-300m-medical"
FAISS_INDEX_PATH = "icd10_faiss.index"
EXCEL_METADATA = "icd10_metadata.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32


# ---------------------
# MODEL SETUP
# ---------------------
def load_model_and_tokenizer(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(DEVICE)
    return tokenizer, model


def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)


def embed_texts(texts, tokenizer, model, batch_size=BATCH_SIZE):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    # L2 normalization for cosine similarity
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


# ---------------------
# INDEX CREATION
# ---------------------
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product for cosine similarity (after normalization)
    index.add(embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved at {FAISS_INDEX_PATH}")
    return index


# ---------------------
# ICD-10 INDEX PIPELINE
# ---------------------
def index_icd10_excel(excel_path):
    df = pd.read_excel(excel_path, dtype=str).fillna("")
    if not {"Code", "Description"}.issubset(df.columns):
        raise ValueError("Excel must contain columns: 'Code' and 'Description'")

    texts = df["Description"].tolist()  # ✅ Only use description for embeddings

    tokenizer, model = load_model_and_tokenizer()
    print("🔹 Generating embeddings...")
    embs = embed_texts(texts, tokenizer, model)

    print("🔹 Building FAISS index...")
    index = build_faiss_index(embs)

    # Save metadata for lookup
    df["faiss_id"] = range(len(df))
    df.to_excel(EXCEL_METADATA, index=False)
    print(f"✅ Metadata saved to {EXCEL_METADATA}")

    # Create mapping: id -> {code, description}
    metadata_map = {
        i: {"code": row["Code"], "description": row["Description"]}
        for i, row in df.iterrows()
    }

    return index, tokenizer, model, metadata_map


# ---------------------
# SEARCH FUNCTION
# ---------------------
def search_icd10(query, index, tokenizer, model, metadata_map, top_k=10, threshold=0.35):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        if score >= threshold:
            meta = metadata_map[idx]
            results.append(
                {
                    "ICD_Code": meta["code"],
                    "Description": meta["description"],
                    "Score (cosine)": round(float(score), 4),
                }
            )
    return results


# ---------------------
# MAIN USAGE EXAMPLE
# ---------------------

excel_path = r"C:\Users\UNegi\OneDrive - Cognitio Analytics LLC\makethon\icd_10_cms_2026.xlsx" # your input Excel
index, tokenizer, model, metadata_map = index_icd10_excel(excel_path)




In [1]:
# ✅ Load FAISS index and metadata
import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
FAISS_INDEX_PATH = r"C:\Users\UNegi\Downloads\icd10_faiss_sapbert_from_pubmedbert.index"
EXCEL_METADATA = r"C:\Users\UNegi\Downloads\icd10_metadata_sapbert_from_pubmedbert.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

# Load FAISS index
index = faiss.read_index(FAISS_INDEX_PATH)

# Load metadata
df = pd.read_excel(EXCEL_METADATA)
metadata_map = {i: {"code": row["Code"], "description": row["Description"]} for i, row in df.iterrows()}

# Embed query
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)


def embed_texts(texts, tokenizer, model, batch_size=BATCH_SIZE):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    # L2 normalization for cosine similarity
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs

def search_icd10(query, index, tokenizer, model, metadata_map, top_k=10, threshold=0.35):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        if score >= threshold:
            meta = metadata_map[idx]
            results.append(
                {
                    "ICD_Code": meta["code"],
                    "Description": meta["description"],
                    "Score (cosine)": round(float(score), 4),
                }
            )
    return results


c:\Users\UNegi\Documents\Project\makethon\.health\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
print(4)

4


In [24]:
import nlu
#Bottle neck give all code due to low blood sugar

ModuleNotFoundError: No module named 'nlu'

In [2]:

query = "Syphliitic corneal inflamation "
results = search_icd10(query, index, tokenizer, model, metadata_map, top_k=10)
print("\n🔍 Search Results:")
for r in results:
    print(r)


🔍 Search Results:
{'ICD_Code': 'A5031   ', 'Description': 'Late congenital syphilitic interstitial keratitis', 'Score (cosine)': 0.7682}
{'ICD_Code': 'A5143   ', 'Description': 'Secondary syphilitic oculopathy', 'Score (cosine)': 0.689}
{'ICD_Code': 'A5271   ', 'Description': 'Late syphilitic oculopathy', 'Score (cosine)': 0.6777}
{'ICD_Code': 'A1852   ', 'Description': 'Tuberculous keratitis', 'Score (cosine)': 0.6593}
{'ICD_Code': 'A5001   ', 'Description': 'Early congenital syphilitic oculopathy', 'Score (cosine)': 0.6524}
{'ICD_Code': 'A5030   ', 'Description': 'Late congenital syphilitic oculopathy, unspecified', 'Score (cosine)': 0.6452}
{'ICD_Code': 'A5144   ', 'Description': 'Secondary syphilitic nephritis', 'Score (cosine)': 0.6397}
{'ICD_Code': 'A5204   ', 'Description': 'Syphilitic cerebral arteritis', 'Score (cosine)': 0.6389}
{'ICD_Code': 'B0052   ', 'Description': 'Herpesviral keratitis', 'Score (cosine)': 0.6314}
{'ICD_Code': 'H15049  ', 'Description': 'Scleritis with c

In [ ]:
cardic

In [ ]:
['H40.05', 'H40.82', 'M10.06', 'M25.06', 'H16.23']


In [ ]:
query = "Hypertension, right eye"
results = search_icd10(query, index, tokenizer, model, metadata_map, top_k=10)

print("\n🔍 Search Results:")
for r in results:
    print(r)

In [9]:

import spacy

from scispacy.abbreviation import AbbreviationDetector

nlp = spacy.load("en_core_sci_sm")

# Add the abbreviation pipe to the spacy pipeline.
nlp.add_pipe("abbreviation_detector")

doc = nlp("Spinal and bulbar muscular atrophy (SBMA) is an \
           inherited motor neuron disease caused by the expansion \
           of a polyglutamine tract within the androgen receptor (AR). \
           SBMA can be caused by this easily.")

print("Abbreviation", "\t", "Definition")
for abrv in doc._.abbreviations:
	print(f"{abrv} \t ({abrv.start}, {abrv.end}) {abrv._.long_form}")


OSError: [E050] Can't find model 'en_core_sci_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:

# Authenticate with Hugging Face before downloading gated models:
#   hf auth login
# Do NOT paste the token into a notebook cell - it ends up in git history.
# Use `hf auth login` interactively, or set the HF_TOKEN environment variable.

In [ ]:
import transformers
import torch

model_id = "ContactDoctor/Bio-Medical-Llama-3-2-1B-CoT-012025"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="cpu",
)

messages = [
    {"role": "system", "content": "You are an expert trained on healthcare and biomedical domain!"},
    {"role": "user", "content": "What are the differential diagnoses for a patient presenting with shortness of breath and chest pain?"},
]

prompt = pipeline.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print(outputs[0]["generated_text"][len(prompt):])


In [ ]:
torch.cuda.is_bf16_supported()

In [ ]:
import transformers
import torch

model_id = "ContactDoctor/Bio-Medical-Llama-3-2-1B-CoT-012025"

# Choose your device explicitly
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else (
    torch.float16 if device == "cuda" else torch.float32
)

pipe = transformers.pipeline(
    task="text-generation",
    model=model_id,
    torch_dtype=dtype,
    device=device,  # <— use this instead of device_map
)

In [ ]:


# 1) Build chat messages
messages = [
    {"role": "system", "content": "You are an expert trained on healthcare and biomedical domain!"},
    {"role": "user", "content": "How similar are both disease Late congenital syphilitic interstitial keratitis and Late-onset congenital syphilitic corneal inflammation"},
]

# 2) Render prompt using the model's chat template
prompt = pipe.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # appends assistant begin token if applicable
)

# 3) Construct terminators (EOS + optional <|eot_id|>)
terminators = []
if pipe.tokenizer.eos_token_id is not None:
    terminators.append(pipe.tokenizer.eos_token_id)
try:
    eot_id = pipe.tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id != pipe.tokenizer.unk_token_id:
        terminators.append(eot_id)
except Exception:
    pass

# 4) Generate
out = pipe(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators if terminators else None,
    do_sample=True,          # set False for deterministic output
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=1.05, # optional: reduce looping
    return_full_text=False,  # only return the generated continuation
)

# 5) Get the assistant's response text
text = out[0]["generated_text"]
print(text)